# Gradientenabstieg

In diesem Notebook können Sie sehen, wie die drei Varianten des Gradientenabstiegs (Gradient Descent, Stochastic Gradient Descent und Mini Batch Gradient Descent) in Python implementiert werden können.
Erinnern Sie sich an die iterativen Schritte des Gradientenabstiegs, um den Code besser zu verstehen.
Es kann Ihnen auch helfen, den Einfluss der Hyperparameter des Gradientenabstiegs zu verstehen, indem Sie versuchen, einige davon hier zu modifizieren.

**Aufgabe:**

Lesen Sie das Notebook durch und versuchen Sie zu verstehen, was bei jedem Schritt passiert. Wie Sie sehen können, fehlen ordnungsgemäße Kommentare, die den Code beschreiben. Fügen Sie diese hinzu, wo immer es notwendig ist (oder hilfreich für Sie, um den Code zu verstehen).

# Gradientenabstieg mit Linearer Regression

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

Mit `plt.style.use` können Sie Layouts (Hintergrundfarbe, Gitter usw.) für das gesamte Notebook festlegen. In diesem Notebook verwenden wir das ggplot-Layout. Es gibt aber noch mehr ([hier klicken](https://matplotlib.org/stable/gallery/style_sheets/style_sheets_reference.html)).

In [ ]:
plt.style.use(['ggplot'])

Wir möchten den Gradientenabstieg verwenden, um optimale Parameter für ein lineares Regressionsproblem zu finden. Hier erstellen wir unsere eigenen Daten, so dass wir wissen, welche Werte die Parameter für eine perfekte Anpassung haben sollten.

$y_{i}=\beta_{0} + \beta_{1}*x_{i} + \varepsilon_{i}$

## Daten Erstellen

#### Generiere einige Daten mit:

$\beta_{0} = 4$ und  $\beta_{1} = 3$

Wir werden auch etwas Gaußsches Rauschen zu den Daten hinzufügen.

In [ ]:
np.random.seed(42)
# create 100 data points for X
X = 2 * np.random.rand(100,1)
# y = b_0 + b_1*X + random noise
y = 4 + 3 * X + np.random.randn(100,1)
print("X dimensions: ", X.shape)
print("y dimensions: ", y.shape)

Lassen Sie uns unsere Daten plotten, um die Beziehung zwischen X und Y zu überprüfen

In [ ]:
plt.plot(X, y, 'b.')
plt.xlabel("$x$", fontsize=18)
plt.ylabel("$y$", rotation=0, fontsize=18);

Dies sind die 100 Datenpunkte, die wir gerade erstellt haben.

## Analytischer Weg der Linearen Regression

Denken Sie daran, dass es eine geschlossene Formel gibt, um die besten Parameter für die lineare Regression zu finden, die **Normal-Gleichung**:

$$ b = (X^TX)^{-1}X^Ty $$

*Zur Erinnerung:* [Wie man Matrizen multipliziert](https://www.mathsisfun.com/algebra/matrix-multiplying.html)

Wir haben diese Formel bereits verwendet, um lineare Regressionsprobleme zu lösen.
Lassen Sie uns sie ausprobieren und sehen, welche Ergebnisse für $b_{0}$ und $b_{1}$ sie liefert.

In [ ]:
# np.c_ can be used to concatenate arrays
X_b = np.c_[np.ones((100,1)), X]
print("X_b dimensions: ", X_b.shape)

# Here we implement the formula above with numpy .T for transpose and .dot for the dot product
# and .linalg.inv for matrix inversion 
b_best = np.linalg.inv(X_b.T.dot(X_b)).dot(X_b.T).dot(y)
print(b_best)

Dies ist nahe an unseren tatsächlichen Parametern 4 und 3. Es kann aufgrund des Rauschens, das wir den Daten hinzugefügt haben, nicht genau sein.

In [ ]:
X_new = np.array([[0],[2]])
X_new_b = np.c_[np.ones((2,1)), X_new]
y_predict = X_new_b.dot(b_best)
y_predict

Lassen Sie uns eine Vorhersagelinie mit den berechneten Parametern plotten.

In [ ]:
plt.plot(X_new, y_predict, 'r-')
plt.plot(X, y, 'b.')
plt.xlabel("$x_1$", fontsize=18)
plt.ylabel("$y$", rotation=0, fontsize=18);

## Gradientenabstieg

Jetzt verwenden wir den Gradientenabstieg, um die optimalen Parameter zu finden.

### Kostenfunktion & Gradienten

Die allgemeinen Gleichungen zur Berechnung der Kostenfunktion und der Gradienten für die lineare Regression sind unten dargestellt.

**Bitte beachten Sie, dass die Kostenfunktion für die lineare Regression gilt. Für andere Algorithmen wird die Kostenfunktion anders sein und die Gradienten müssten aus den Kostenfunktionen abgeleitet werden.**


**Kosten**

$$J(b) = \frac{ 1 }{ 2n } \sum_{i=1}^{n} (\hat{y}_i - y_i)^2$$

**Gradient**

$$ \frac{\partial J(b)}{\partial b_j} = \frac{ 1 }{ n }\sum_{i=1}^{n}(\hat{y}_i - y_i) x_{i,j} $$

**Aktualisierung der Parameter mit den Gradienten**

$$ b_0: = b_0 -\alpha (\frac{ 1 }{ n } \sum_{i=1}^{n}(\hat{y}_i - y_i) * 1) $$

$$ b_1: = b_1 -\alpha (\frac{ 1 }{ n } \sum_{i=1}^{n}(\hat{y}_i - y_i)  x_{i,1})$$

$$ b_2: = b_2 -\alpha (\frac{ 1 }{ n } \sum_{i=1}^{n}(\hat{y}_i - y_i) x_{i,2})$$

$$ b_j: = b_j -\alpha (\frac{ 1 }{ n } \sum_{i=1}^{n}(\hat{y}_i - y_i) x_{i,j})$$


Als nächstes werden die drei Varianten der Gradientenabstiegsmethode angewendet: **Batch Gradient Descent**, **Stochastic Gradient Descent** und **Mini-Batch Gradient Descent**. Erinnern Sie sich daran, welche Schritte im Gradientenabstiegsverfahren iterativ wiederholt werden.

## Batch Gradient Descent

Batch Gradient Descent berechnet den Gradienten der Kostenfunktion in Bezug auf die Parameter $ b $ für den gesamten Trainingsdatensatz:

$$ b = b - \alpha \frac{\partial}{\partial b} J(b) $$

Da wir die Gradienten für den gesamten Datensatz berechnen müssen, um nur eine Aktualisierung durchzuführen, kann Batch Gradient Descent sehr langsam sein und ist für Datensätze, die nicht in den Speicher passen, nicht praktikabel. Batch Gradient Descent erlaubt es uns auch nicht, unser Modell online zu aktualisieren, d.h. mit neuen Beispielen im laufenden Betrieb.

Wir aktualisieren dann unsere Parameter in die entgegengesetzte Richtung der Gradienten, wobei die Lernrate bestimmt, wie groß eine Aktualisierung wir durchführen. Batch Gradient Descent konvergiert garantiert zum globalen Minimum für konvexe Fehleroberflächen und zu einem lokalen Minimum für nicht-konvexe Oberflächen.

In [ ]:
def cal_cost(b, X, y):
    """
    Compute the mean squared error cost for linear regression.

    Parameters
    ----------
    b : ndarray of shape (j, 1)
        Parameter vector (including bias weight).
    X : ndarray of shape (n, j)
        Feature matrix including the bias column (all ones).
    y : ndarray of shape (n, 1)
        True target values.

    Returns
    -------
    cost : float
        The mean squared error cost.
    """

    n = y.shape[0]
    
    predictions = X.dot(b)
    cost = 1/(2*n) * np.sum(np.square(predictions-y))
    return cost

In [ ]:
def batch_gradient_descent(X, y, b, learning_rate=0.01, iterations=1000):
    """
        Perform batch gradient descent for linear regression.

        Parameters
        ----------
        X : ndarray of shape (n, j)
            Feature matrix including bias column.
        y : ndarray of shape (n, 1)
            Target values.
        b : ndarray of shape (j, 1)
            Initial parameter vector.
        learning_rate : float, optional
            Step size for parameter updates.
        iterations : int, optional
            Number of gradient descent iterations.

        Returns
        -------
        b : ndarray of shape (j, 1)
            Final learned parameter vector.
        cost_history : ndarray of shape (iterations,)
            Cost value at each iteration.
        b_history : ndarray of shape (iterations, j)
            Parameter values at each iteration.
    """
    
    n = y.shape[0]
    cost_history = np.zeros(iterations)
    b_history = np.zeros((iterations, b.shape[0]))
    
    for it in range(iterations):
        
        prediction = np.dot(X,b)
        
        b = b - learning_rate * (1/n) * (X.T.dot((prediction - y)))
        b_history[it,:] = b.T
        cost_history[it]  = cal_cost(b, X, y)
        
    return b, cost_history, b_history

Beginnen wir mit 1000 Iterationen und einer Lernrate von 0,01. Beginnen Sie mit initialen Parametern aus einer Gaußverteilung.

In [ ]:
lr = 0.01
n_iter = 1000

b = np.random.randn(2,1)

X_b = np.c_[np.ones((len(X),1)),X]
b, cost_history, b_history = batch_gradient_descent(X_b, y, b, lr, n_iter)


print(f"b_0:             {b[0, 0]:0.3f},\n"
      f"b_1:             {b[1, 0]:0.3f}")

print(f"Final cost:      {cost_history[-1]:0.3f}")

Lassen Sie uns den Kostenverlauf über die Iterationen plotten:

In [ ]:
fig, ax = plt.subplots(figsize=(12,8))

ax.set_ylabel('J$(b)$')
ax.set_xlabel('Iterations')
ax.plot(range(n_iter), cost_history, 'b.');

Nach etwa 150 Iterationen ist die Kostenfunktion flach, so dass die verbleibenden Iterationen nicht benötigt werden oder zu keiner weiteren Optimierung führen. Lassen Sie uns bis zur Iteration 200 hineinzoomen und die Kurve betrachten.

In [ ]:
fig,ax = plt.subplots(figsize=(10,8))
ax.set_ylabel('J$(b)$')
ax.set_xlabel('Iterations')
ax.plot(range(200), cost_history[:200], 'b.');

Es ist erwähnenswert, dass die Kosten anfangs schneller sinken und dann der Gewinn an Kostenreduzierung nicht mehr so groß ist.

Es wäre großartig, die Auswirkungen verschiedener Lernraten und Iterationen zusammen zu sehen. Lassen Sie uns eine Funktion erstellen, die die Effekte zusammen zeigen kann und auch zeigt, wie der Gradientenabstieg tatsächlich funktioniert.

In [ ]:
def plot_GD(n_iter,lr, ax, ax1=None):
    """
    Plot gradient descent steps for linear regression.

    Parameters
    ----------
    n_iter : int
        Number of gradient descent iterations.
    lr : float
        Learning rate.
    ax : matplotlib.axes.Axes
        Axis for plotting data points and prediction lines.
    ax1 : matplotlib.axes.Axes, optional
        Axis for plotting cost vs iterations.
    """
    
    ax.plot(X, y, 'b.')
    b = np.random.randn(2,1)

    transparency = 0.1
    cost_history = np.zeros(n_iter)
    for i in range(n_iter):
        #pred_prev = X_b.dot(b)
        b, h, _ = batch_gradient_descent(X_b, y, b, lr, 1)
        pred = X_b.dot(b)

        cost_history[i] = h[0]

        if i % 25 == 0:
            ax.plot(X, pred, 'r-', alpha=transparency)
            if transparency < 0.8:
                transparency = transparency+0.2
    if ax1 is not None:
        ax1.plot(range(n_iter), cost_history, 'b.')  

Plotten Sie die Graphen für verschiedene Kombinationen von Iterationen und Lernraten.

In [ ]:
fig = plt.figure(figsize=(30,25), dpi=200)
fig.subplots_adjust(hspace=0.4, wspace=0.4)

iterations_lr = [(2000,0.001), (500,0.01), (200,0.05), (100,0.1)]

for i, (n_iter, lr) in enumerate(iterations_lr):
    ax  = fig.add_subplot(4, 2, 2*i + 1)
    ax1 = fig.add_subplot(4, 2, 2*i + 2)

    ax.set_title(f"lr: {lr}")
    ax1.set_title(f"Iterations: {n_iter}")

    plot_GD(n_iter, lr, ax, ax1)


Sehen Sie, wie nützlich es ist, die Auswirkungen von Lernraten und Iterationen auf den Gradientenabstieg zu visualisieren. Die roten Linien zeigen, wie der Gradientenabstieg beginnt und dann langsam näher an den Endwert herankommt.

Sie können immer einzelne Graphen plotten, um hineinzuzoomen:

In [ ]:
_, ax = plt.subplots(figsize=(14,10))
plot_GD(100, 0.1, ax)

## Stochastic Gradient Descent

Stochastic Gradient Descent (SGD) führt im Gegensatz dazu eine Parameteraktualisierung für jedes Trainingsbeispiel $ x_i $ und Label $ y_i $ durch:

$$ b = b - \alpha \frac{\partial}{\partial b} J(b; x_i; y_i ) $$

Batch Gradient Descent führt redundante Berechnungen für große Datensätze durch, da es Gradienten für ähnliche Beispiele vor jeder Parameteraktualisierung neu berechnet. SGD beseitigt diese Redundanz, indem es jeweils eine Aktualisierung durchführt. Es ist daher in der Regel viel schneller und kann auch für Online-Lernen verwendet werden.

Während Batch Gradient Descent zum Minimum des Beckens konvergiert, in dem die Parameter platziert sind, ermöglicht die Fluktuation von SGD einerseits, zu neuen und potenziell besseren lokalen Minima zu springen. Andererseits erschwert dies letztendlich die Konvergenz zum exakten Minimum, da SGD immer wieder überschießen wird. Es wurde jedoch gezeigt, dass SGD, wenn wir die Lernrate langsam verringern, das gleiche Konvergenzverhalten wie Batch Gradient Descent zeigt und mit ziemlicher Sicherheit zu einem lokalen oder dem globalen Minimum für nicht-konvexe bzw. konvexe Optimierung konvergiert.

Fühlen Sie sich frei, einige Parameter zu ändern und zu sehen, was passiert.

In [ ]:
def stochastic_gradient_descent(X, y, b, learning_rate=0.01, epochs=10):
    """
    Perform stochastic gradient descent for linear regression.

    Parameters
    ----------
    X : ndarray of shape (n, j)
        Feature matrix with bias column.
    y : ndarray of shape (n, 1)
        Target values.
    b : ndarray of shape (j, 1)
        Initial parameter vector.
    learning_rate : float
        Learning rate.
    epochs : int
        Number of passes over the training data.

    Returns
    -------
    b : ndarray of shape (j, 1)
        Final parameter vector.
    cost_history : ndarray of shape (epochs,)
        Cost value at the end of each epoch.
    """
    n = X.shape[0]
    cost_history = np.zeros(epochs)
    
    
    for epoch in range(epochs):
        cost = 0.0
        for _ in range(n):
            rand_ind = np.random.randint(0, n)
            #print("rand_ind:", rand_ind)
            X_i = X[rand_ind,:].reshape(1, X.shape[1])
            #print("X_i shape:", X_i.shape)
            y_i = y[rand_ind].reshape(1, 1)
            #print("y_i shape:", y_i.shape)
            prediction = np.dot(X_i, b)

            b = b - (1/1) * learning_rate * (X_i.T.dot((prediction - y_i)))
            cost = cost +  cal_cost(b, X_i, y_i)
        cost_history[epoch]  = cost/n
        
    return b, cost_history


In [ ]:
lr = 0.01
epochs = 2000

b = np.random.randn(2,1)
X_b = np.c_[np.ones((len(X),1)), X]
b, cost_history = stochastic_gradient_descent(X_b, y, b, lr, epochs)

print(f'b_0: {b[0][0]:0.3f},\nb_1: {b[1][0]:0.3f}')
print(f'Final cost  {cost_history[-1]:0.3f}')

In [ ]:
fig, ax = plt.subplots(figsize=(10,8))

ax.set_ylabel('J$(b)$', rotation=0)
ax.set_xlabel('Epochs')
b = np.random.randn(2,1)

ax.plot(range(epochs), cost_history, 'b.');

## Mini Batch Gradient Descent

Mini-Batch Gradient Descent nimmt schließlich das Beste aus beiden Welten und führt eine Aktualisierung für jeden Mini-Batch von $ m $ Trainingsbeispielen durch.

$$ b = b - \alpha \frac{\partial}{\partial b} J(b; x_{(i:i+m)}; y_{(i:i+m)} ) $$

Auf diese Weise:
1. reduziert es die Varianz der Parameteraktualisierungen, was zu einer stabileren Konvergenz führen kann
2. kann es hochoptimierte Matrixoptimierungen nutzen, die in modernen Deep-Learning-Bibliotheken üblich sind und die Berechnung des Gradienten in Bezug auf einen Mini-Batch sehr effizient machen.

Übliche Mini-Batch-Größen liegen zwischen 50 und 256, können aber je nach Anwendung variieren. Mini-Batch Gradient Descent ist typischerweise der Algorithmus der Wahl beim Training eines neuronalen Netzes, und der Begriff SGD wird normalerweise auch verwendet, wenn Mini-Batches verwendet werden.

In [ ]:
def minibatch_gradient_descent(X, y, b, learning_rate=0.01, epochs=10, batch_size=20):
    """
    Perform mini-batch gradient descent for linear regression.

    Parameters
    ----------
    X : ndarray of shape (n, j)
        Feature matrix without bias column.
    y : ndarray of shape (n, 1)
        Target values.
    b : ndarray of shape (j, 1)
        Initial parameter vector.
    learning_rate : float
        Learning rate.
    epochs : int
        Number of passes over the training data.
    batch_size : int
        Number of samples per mini-batch.

    Returns
    -------
    b : ndarray of shape (j, 1)
        Final learned parameter vector.
    cost_history : ndarray of shape (epochs,)
        True cost on the FULL dataset at the end of each epoch.
    """
    n = X.shape[0]
    cost_history = np.zeros(epochs)
    
    for epoch in range(epochs):
        cost = 0.0
        shuffled_indices = np.random.permutation(n)
        X = X[shuffled_indices]
        y = y[shuffled_indices]
        for ind, i in enumerate(range(0, n, batch_size)):
            print(f"iteration nr:{ind},in epoch nr:{epoch}")
            X_i = X[i:i+batch_size]
            y_i = y[i:i+batch_size]
            
            X_i = np.c_[np.ones(len(X_i)),X_i]
           
            prediction = np.dot(X_i, b)

            b = b - (1/batch_size) * learning_rate * (X_i.T.dot((prediction - y_i)))
            cost += cal_cost(b, X_i, y_i)
        cost_history[epoch]  = cost/(ind+1)     # cost/number of batches
        
    return b, cost_history

In [ ]:
lr = 0.01
epochs = 2000

b = np.random.randn(2,1)


b, cost_history = minibatch_gradient_descent(X, y, b, lr, epochs, batch_size=20)


print(f'b_0: {b[0][0]:0.3f},\nb_1: {b[1][0]:0.3f}')
print(f'Final cost:  {cost_history[-1]:0.3f}')

In [ ]:
fig, ax = plt.subplots(figsize=(10,8))

ax.set_ylabel('J$(b)$', rotation=0)
ax.set_xlabel('Epochs')
theta = np.random.randn(2,1)

ax.plot(range(epochs), cost_history, 'b.');